# Chapter 22 — Retrieval Is Not Geometry

**Book alignment:** *Embeddings From First Principles*, Chapter 22

**Notebook role:** `SYNTHETIC_DEMO + ARTIFACT_REPLAY` — sections 1-6 construct a controlled toy example of the mechanism (reproducing no benchmark); the final section replays the frozen Wave 6 artifact and re-derives the numbers the chapter quotes.

**Question this notebook isolates:** Can a translated point recover its paired target almost
perfectly while failing to preserve the target model's local neighborhood?

This notebook implements the chapter's **reader lab as a deterministic synthetic experiment**.
It does **not** reproduce the chapter's measured Wave 6 mxbai→Qwen3 benchmark (wave6/artifacts/cross-space-benchmark.json). The point is
to construct a controlled counterexample that you can modify and inspect.

In [1]:
import numpy as np

rng = np.random.default_rng(22)

N = 200
D = 32
K = 10
N_CLUSTERS = 8
NOISE = 0.22


def normalize_rows(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)


def cosine_matrix(x, y):
    return normalize_rows(x) @ normalize_rows(y).T

## 1. Build a target space with crowded local neighborhoods

We create 200 points in eight loose clusters. Nearby items are intentionally close enough that
a modest anisotropic distortion can reorder many near-ties without losing the paired target.

In [2]:
centers = rng.normal(size=(N_CLUSTERS, D))
centers = normalize_rows(centers)

labels = np.repeat(np.arange(N_CLUSTERS), N // N_CLUSTERS)
target = np.vstack([
    centers[label] + NOISE * rng.normal(size=D)
    for label in labels
])
target = normalize_rows(target)

# Create a source space by a pure orthogonal basis change.
q, _ = np.linalg.qr(rng.normal(size=(D, D)))
source = target @ q

print("target shape:", target.shape)
print("source shape:", source.shape)
print("same objects, different coordinates")

target shape: (200, 32)
source shape: (200, 32)
same objects, different coordinates


## 2. Define the two contracts

**Counterpart recovery** asks where the translated version of object *i* ranks its native target
vector *i*.

**Structural fidelity** asks whether the translated query sees the same target-space neighbors
that the native target vector sees.

In [3]:
native_similarity = cosine_matrix(target, target)


def topk_without_self(scores, i, k=K):
    return [j for j in np.argsort(-scores) if j != i][:k]


native_neighbors = [
    topk_without_self(native_similarity[i], i, K)
    for i in range(N)
]


def evaluate(translated, k=K):
    translated = normalize_rows(translated)
    cross = cosine_matrix(translated, target)

    # Counterpart recovery: paired target i is a candidate.
    ranking = np.argsort(-cross, axis=1)
    paired_rank = np.array([
        np.where(ranking[i] == i)[0][0]
        for i in range(N)
    ])

    recall_at_1 = np.mean(paired_rank < 1)
    recall_at_10 = np.mean(paired_rank < 10)
    mrr = np.mean(1.0 / (paired_rank + 1))
    paired_cosine = np.mean(np.sum(translated * target, axis=1))

    # Structural fidelity: compare neighborhoods, excluding the paired item itself.
    translated_neighbors = [
        topk_without_self(cross[i], i, k)
        for i in range(N)
    ]
    agreement = np.mean([
        len(set(native_neighbors[i]) & set(translated_neighbors[i])) / k
        for i in range(N)
    ])

    # Rank correlation only over shared neighbors.
    rank_correlations = []
    for i in range(N):
        shared = list(set(native_neighbors[i]) & set(translated_neighbors[i]))
        if len(shared) < 2:
            rank_correlations.append(0.0)
            continue

        native_rank = np.array([native_neighbors[i].index(j) for j in shared], dtype=float)
        translated_rank = np.array([translated_neighbors[i].index(j) for j in shared], dtype=float)

        native_rank -= native_rank.mean()
        translated_rank -= translated_rank.mean()
        denom = np.sqrt((native_rank ** 2).sum() * (translated_rank ** 2).sum())
        rank_correlations.append(
            float(native_rank @ translated_rank / denom) if denom else 0.0
        )

    return {
        "paired_cosine": float(paired_cosine),
        "recall@1": float(recall_at_1),
        "recall@10": float(recall_at_10),
        "mrr": float(mrr),
        "agreement@10": float(agreement),
        "shared_neighbor_rank_corr": float(np.mean(rank_correlations)),
        "paired_rank": paired_rank,
        "translated_neighbors": translated_neighbors,
    }

## 3. Start from a perfect bridge, then distort it

Because `source = target @ Q`, the exact inverse bridge is `Q.T`.

We then add a controlled anisotropic scaling **after** the correct translation. This keeps each
point near its paired target while changing which dimensions dominate local geometry.

In [4]:
perfect = source @ q.T
perfect_report = evaluate(perfect)

print("perfect bridge")
for key in ("paired_cosine", "recall@10", "mrr", "agreement@10", "shared_neighbor_rank_corr"):
    print(f"  {key:28} {perfect_report[key]:.3f}")

assert perfect_report["recall@10"] == 1.0
assert perfect_report["agreement@10"] == 1.0

perfect bridge
  paired_cosine                1.000
  recall@10                    1.000
  mrr                          1.000
  agreement@10                 1.000
  shared_neighbor_rank_corr    1.000


In [5]:
def distorted_bridge(alpha):
    scales = np.ones(D)
    scales[:8] = 1.0 + alpha
    scales[8:16] = 1.0 / (1.0 + 0.5 * alpha)

    distortion = np.diag(scales)
    translated = (source @ q.T) @ distortion
    return normalize_rows(translated)


rows = []
for alpha in np.linspace(0.0, 3.0, 13):
    report = evaluate(distorted_bridge(alpha))
    rows.append((alpha, report["recall@10"], report["agreement@10"], report["paired_cosine"]))

print(f"{'alpha':>6} {'R@10':>8} {'agree@10':>10} {'paired cos':>11}")
for alpha, r10, agree, cos in rows:
    print(f"{alpha:6.2f} {r10:8.3f} {agree:10.3f} {cos:11.3f}")

 alpha     R@10   agree@10  paired cos
  0.00    1.000      1.000       1.000
  0.25    1.000      0.916       0.992
  0.50    1.000      0.857       0.974
  0.75    1.000      0.805       0.950
  1.00    1.000      0.758       0.924
  1.25    1.000      0.718       0.898
  1.50    1.000      0.676       0.873
  1.75    1.000      0.648       0.850
  2.00    1.000      0.625       0.829
  2.25    1.000      0.606       0.809
  2.50    1.000      0.586       0.791
  2.75    1.000      0.573       0.775
  3.00    1.000      0.556       0.760


## 4. Find the counterexample automatically

The lab's success condition is:

- paired-target `Recall@10 >= 0.95`
- native-vs-translated `agreement@10 < 0.70`

That is enough to falsify the claim:

> "The bridge retrieves the right point, therefore it preserved the space."

In [6]:
chosen = None

for alpha in np.linspace(0.0, 3.0, 121):
    report = evaluate(distorted_bridge(alpha))
    if report["recall@10"] >= 0.95 and report["agreement@10"] < 0.70:
        chosen = (alpha, report)
        break

assert chosen is not None

alpha, report = chosen
print(f"first distortion satisfying the lab criterion: alpha={alpha:.3f}")
for key in ("paired_cosine", "recall@1", "recall@10", "mrr", "agreement@10", "shared_neighbor_rank_corr"):
    print(f"  {key:28} {report[key]:.3f}")

assert report["recall@10"] >= 0.95
assert report["agreement@10"] < 0.70

print("\nCounterexample established:")
print("paired-target recovery survives while local target geometry does not.")

first distortion satisfying the lab criterion: alpha=1.375
  paired_cosine                0.886
  recall@1                     1.000
  recall@10                    1.000
  mrr                          1.000
  agreement@10                 0.698
  shared_neighbor_rank_corr    0.416

Counterexample established:
paired-target recovery survives while local target geometry does not.


## 5. Inspect one object where the paired target survives but neighbors churn

This makes the aggregate result concrete.

In [7]:
translated = distorted_bridge(alpha)
cross = cosine_matrix(translated, target)

example = None
for i in range(N):
    native = native_neighbors[i]
    translated_n = report["translated_neighbors"][i]
    overlap = len(set(native) & set(translated_n))
    paired_position = int(np.where(np.argsort(-cross[i]) == i)[0][0])

    if paired_position == 0 and overlap <= 6:
        example = (i, native, translated_n, overlap)
        break

assert example is not None

i, native, translated_n, overlap = example
print("object:", i)
print("paired target rank:", 1)
print("native target neighbors:     ", native)
print("translated-query neighbors:  ", translated_n)
print(f"overlap: {overlap}/{K}")

lost = [j for j in native if j not in translated_n]
gained = [j for j in translated_n if j not in native]
print("lost neighbors:  ", lost)
print("gained neighbors:", gained)

object: 1
paired target rank: 1
native target neighbors:      [np.int64(19), np.int64(10), np.int64(5), np.int64(12), np.int64(18), np.int64(24), np.int64(13), np.int64(154), np.int64(6), np.int64(7)]
translated-query neighbors:   [np.int64(5), np.int64(10), np.int64(6), np.int64(12), np.int64(150), np.int64(31), np.int64(27), np.int64(154), np.int64(19), np.int64(17)]
overlap: 6/10
lost neighbors:   [np.int64(18), np.int64(24), np.int64(13), np.int64(7)]
gained neighbors: [np.int64(150), np.int64(31), np.int64(27), np.int64(17)]


## 6. Turn the measurement into a consumer decision

A high-recovery / low-structure bridge can still be useful. The verdict depends on what the
consumer needs.

In [8]:
consumer_requirements = {
    "legacy_record_lookup": "point_recovery",
    "candidate_generation_then_native_rerank": "point_recovery",
    "ranked_search_ui": "ranking",
    "clustering": "neighborhood",
    "neighbor_sensitive_dedup": "neighborhood",
}

def consumer_verdict(requirement, report):
    if requirement == "point_recovery":
        return report["recall@10"] >= 0.95
    if requirement == "ranking":
        return report["agreement@10"] >= 0.80 and report["shared_neighbor_rank_corr"] >= 0.70
    if requirement == "neighborhood":
        return report["agreement@10"] >= 0.80
    raise ValueError(requirement)

for consumer, requirement in consumer_requirements.items():
    verdict = consumer_verdict(requirement, report)
    print(f"{consumer:38} requires={requirement:15} {'PASS' if verdict else 'FAIL'}")

legacy_record_lookup                   requires=point_recovery  PASS
candidate_generation_then_native_rerank requires=point_recovery  PASS
ranked_search_ui                       requires=ranking         FAIL
clustering                             requires=neighborhood    FAIL
neighbor_sensitive_dedup               requires=neighborhood    FAIL


## What we earned

This synthetic construction demonstrates the chapter's central distinction:

- **Counterpart recovery** asks whether the translated vector finds the native target for the same
  object.
- **Structural fidelity** asks whether the translated vector reproduces the target space's
  relationships.
- High paired-target Recall@10 does **not** imply high neighborhood agreement.
- A bridge can therefore be usable for lookup or migration while failing consumers that rely on
  rank, neighborhoods, or clustering.
- Direction, metric family, and consumer contract belong in the preservation claim.

The chapter's published manuscript reports a separate **measured** mxbai→Qwen3 result
(Recall@10 0.997 vs agreement@10 0.589). This notebook is a controlled reader exercise, not a
reproduction of that benchmark.

**Next:** Chapter 23 asks a deeper question: if source and target geometry disagree, **whose
geometry should the translation preserve?**

## 7. Replay the measured Wave 6 benchmark

The construction above is synthetic. This section reads the frozen Wave 6 artifact
(`experiments/embeddings-from-first-principles/wave6/artifacts/cross-space-benchmark.json`)
and checks the headline numbers the chapter quotes for the real
`mxbai-embed-large -> Qwen3-Embedding-8B` bridge.


In [ ]:
from pathlib import Path
import json


def _wave6():
    for c in (Path.cwd(), *Path.cwd().parents):
        p = c / 'experiments/embeddings-from-first-principles/wave6/artifacts/cross-space-benchmark.json'
        if p.exists():
            return json.loads(p.read_text(encoding='utf-8'))
    raise RuntimeError('run from a checkout with the wave6 artifact')


w6 = _wave6()
fwd = w6['maps']['mxbai->qwen ridge']
ctl = w6['maps']['mxbai->bge-large ridge (control)']
rev = w6['maps']['qwen->mxbai ridge']

print('corpus', w6['corpus']['n_items'], 'sentences, split', w6['corpus']['split'])
print('native CKA  mxbai-Qwen', w6['native_geometry']['mxbai_vs_qwen_cka'],
      ' mxbai-bge', w6['native_geometry']['mxbai_vs_bge_cka'])
for name, m in [('mxbai -> Qwen', fwd), ('control mxbai -> bge', ctl), ('reverse Qwen -> mxbai', rev)]:
    print(f"{name:22s}  R@1 {m['recall_at_1']:.3f}  R@10 {m['recall_at_10']:.3f}  "
          f"agree@10 {m['agreement_at_10']:.3f}  order {m['order_preservation_local']:.2f}  "
          f"ARI {m['cluster_preservation_ari']:.3f}")

# counterpart recovery is near-perfect; structural fidelity is clearly lower
assert fwd['recall_at_10'] == 1.0 and fwd['recall_at_1'] > 0.8
assert fwd['agreement_at_10'] < 0.85 and fwd['order_preservation_local'] < 0.8
# the near-same-space control preserves structure the divergent pair loses
assert ctl['agreement_at_10'] > fwd['agreement_at_10'] + 0.08
assert ctl['order_preservation_local'] > fwd['order_preservation_local'] + 0.15
# direction changes the profile
assert abs(rev['cluster_preservation_ari'] - fwd['cluster_preservation_ari']) > 0.05
print('counterpart recovery >> structural fidelity; the control preserves both')
